Now given new team data, I want to see how a logistic regressor performs on the base data before any feature engineering

In [2]:
import pandas as pd
df = pd.read_csv("../../data/aggregated_rolling_features.csv")
df.shape

(5539, 97)

Drop arbitrary data like id and names

In [3]:
to_drop = []
to_drop.extend([f"team{i}_player{k}_id" for i in range(1,3) for k in range(1,6)])
to_drop.extend([f"team{i}_id" for i in range(1,3)])
to_drop.extend([f"team{i}" for i in range(1,3)])
to_drop.extend(["tournament", "match_id", "game_id", "map_id", "map_name", "datetime", "Unnamed: 0"])
df = df.drop(to_drop, axis=1)
df

,team1_win,bestOf,team1_previous_10_average_map_score,team2_previous_10_average_map_score,previous_10_games_team1_average_kills,previous_10_games_team1_std_kills,previous_10_games_team1_range_kills,previous_10_games_team1_max_kills,previous_10_games_team1_min_kills,previous_10_games_team1_median_kills,...,previous_10_games_team2_range_kast,previous_10_games_team2_max_kast,previous_10_games_team2_min_kast,previous_10_games_team2_median_kast,previous_10_games_team2_average_kddiff,previous_10_games_team2_std_kddiff,previous_10_games_team2_range_kddiff,previous_10_games_team2_max_kddiff,previous_10_games_team2_min_kddiff,previous_10_games_team2_median_kddiff
0,0,3.0,13.000000,13.000000,16.000000,0.000000e+00,0.0,16.000000,16.000000,16.000000,...,0.00,70.000000,70.000000,70.000000,1.000000,0.000000,0.0,1.000000,1.000000,1.000000
1,1,3.0,12.333333,12.333333,15.600000,4.127953e+00,10.0,22.000000,12.000000,13.000000,...,25.00,83.300000,58.300000,58.300000,-0.600000,3.136877,9.0,4.000000,-5.000000,0.000000
2,1,3.0,11.600000,11.600000,15.100000,2.083267e+00,5.5,18.500000,13.000000,14.000000,...,17.30,75.000000,57.700000,60.100000,-1.600000,5.885576,17.5,8.500000,-9.000000,-1.500000
3,1,3.0,11.571429,11.571429,14.612903,1.776357e-15,0.0,14.612903,14.612903,14.612903,...,0.00,71.535484,71.535484,71.535484,-0.032258,0.000000,0.0,-0.032258,-0.032258,-0.032258
4,1,3.0,10.666667,10.666667,13.600000,4.363485e+00,13.0,21.000000,8.000000,13.000000,...,33.30,60.000000,26.700000,33.300000,-9.000000,2.280351,6.0,-6.000000,-12.000000,-9.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5534,0,3.0,13.400000,9.500000,14.140000,9.200000e-01,2.6,15.400000,12.800000,14.200000,...,6.89,77.400000,70.510000,76.760000,0.780000,2.465279,6.4,3.500000,-2.900000,1.800000
5535,0,3.0,9.700000,11.100000,14.680000,1.175415e+00,3.3,16.600000,13.300000,14.900000,...,6.96,78.300000,71.340000,76.820000,0.920000,2.741095,7.0,3.700000,-3.300000,2.300000
5536,1,3.0,11.000000,10.700000,14.440000,6.590903e-01,1.7,15.400000,13.700000,14.600000,...,6.96,79.800000,72.840000,75.820000,1.140000,2.793278,7.1,3.900000,-3.200000,2.700000
5537,0,5.0,11.400000,12.700000,14.780000,2.594147e+00,6.8,17.500000,10.700000,14.900000,...,5.71,83.730000,78.020000,82.000000,4.280000,3.398470,8.7,7.800000,-0.900000,5.200000


In [4]:
X = df.drop("team1_win", axis=1)
y = df.team1_win

In [5]:
X_numerical = X.drop("bestOf", axis=1)
X_categorical = X.bestOf

Standardize and one-hot encode

In [6]:
from sklearn.preprocessing import StandardScaler
from math import ceil
stnd = StandardScaler().set_output(transform="pandas")
split_point = ceil(len(df) * 0.8)
train_numerical = X_numerical.iloc[:split_point] # 0 to split_point - 1
test_numerical = X_numerical.iloc[split_point:] # split_point to len(df)
train_cat = X_categorical.iloc[:split_point]
test_cat = X_categorical.iloc[split_point:]
y_train = y.iloc[:split_point]
y_test = y.iloc[split_point:]
train_numerical = stnd.fit_transform(train_numerical)
test_numerical = stnd.transform(test_numerical)
train_cat = pd.get_dummies(train_cat, prefix="bestOf", drop_first=True, dtype=int)
test_cat = pd.get_dummies(test_cat, prefix="bestOf", drop_first=True, dtype=int)
X_train = pd.concat([train_numerical, train_cat], axis=1)
X_test = pd.concat([test_numerical, test_cat], axis=1)
display(X_train)
display(X_test)

,team1_previous_10_average_map_score,team2_previous_10_average_map_score,previous_10_games_team1_average_kills,previous_10_games_team1_std_kills,previous_10_games_team1_range_kills,previous_10_games_team1_max_kills,previous_10_games_team1_min_kills,previous_10_games_team1_median_kills,previous_10_games_team1_average_deaths,previous_10_games_team1_std_deaths,...,previous_10_games_team2_min_kast,previous_10_games_team2_median_kast,previous_10_games_team2_average_kddiff,previous_10_games_team2_std_kddiff,previous_10_games_team2_range_kddiff,previous_10_games_team2_max_kddiff,previous_10_games_team2_min_kddiff,previous_10_games_team2_median_kddiff,bestOf_3.0,bestOf_5.0
0,1.067406,1.103525,1.191579,-2.267983,-2.232999,-0.412393,2.477353,1.117133,0.481263,-2.420802,...,0.563877,-0.400940,0.744679,-2.358334,-2.318953,-0.792708,2.108424,0.709155,1,0
1,0.694554,0.753637,0.887984,3.156345,2.427842,2.609393,-0.017376,-1.000761,0.481263,4.429077,...,-1.840127,-3.414365,-0.299688,1.205328,1.300430,0.550491,-1.057503,0.108085,1,0
2,0.284417,0.368761,0.508491,0.469529,0.330464,0.846685,0.606306,-0.294796,-0.534503,3.051541,...,-1.963409,-2.950761,-0.952418,4.327997,4.718737,2.565289,-3.168122,-0.793522,1,0
3,0.268438,0.353766,0.138792,-2.267983,-2.232999,-1.110977,1.612245,0.137892,0.223811,-2.420802,...,0.879373,-0.005464,0.070893,-2.358334,-2.318953,-1.254884,1.563748,0.088695,1,0
4,-0.237576,-0.121081,-0.629988,3.465844,3.826094,2.105762,-2.512105,-1.000761,-7.064424,1.510645,...,-8.332992,-9.853307,-5.782617,0.232268,0.093969,-3.926839,-4.751085,-5.301553,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4427,1.067406,-0.051104,0.204897,1.381192,1.449065,0.846685,-0.890531,0.411169,-0.621568,0.051191,...,1.854231,2.058736,1.867374,-1.409474,-1.313569,0.640037,2.477782,1.610762,1,0
4428,0.955551,1.313457,0.144178,-0.666354,-0.555096,-0.362030,0.294465,0.481765,-0.708634,0.535262,...,0.818660,1.438022,1.671555,-0.043013,-0.469046,1.042997,1.844597,0.769263,1,0
4429,0.060706,0.526211,-0.022799,-0.468170,-0.368663,-0.412393,-0.017376,0.199379,-0.636079,0.645259,...,0.880301,1.237127,1.632391,-0.470159,-0.629907,0.998224,2.002893,1.009691,1,0
4430,-0.218933,0.368761,-0.402292,-0.591349,-0.601705,-0.865660,-0.266849,-0.012410,-1.216517,0.286723,...,0.775511,0.485059,1.332135,-0.211222,-0.026677,0.998224,1.211411,0.889477,1,0


,team1_previous_10_average_map_score,team2_previous_10_average_map_score,previous_10_games_team1_average_kills,previous_10_games_team1_std_kills,previous_10_games_team1_range_kills,previous_10_games_team1_max_kills,previous_10_games_team1_min_kills,previous_10_games_team1_median_kills,previous_10_games_team1_average_deaths,previous_10_games_team1_std_deaths,...,previous_10_games_team2_min_kast,previous_10_games_team2_median_kast,previous_10_games_team2_average_kddiff,previous_10_games_team2_std_kddiff,previous_10_games_team2_range_kddiff,previous_10_games_team2_max_kddiff,previous_10_games_team2_min_kddiff,previous_10_games_team2_median_kddiff,bestOf_3.0,bestOf_5.0
4432,0.578038,-0.018424,0.538851,0.558411,0.423680,0.040875,-0.516322,1.328923,-0.171729,0.195478,...,-0.272388,-1.098921,-0.599944,-1.609572,-1.554861,-1.240441,0.578226,-0.432879,1,0
4433,0.060706,0.158829,-0.538910,-0.096979,-0.089012,-0.311666,-0.266849,-0.153603,-2.000107,-0.582787,...,0.093350,0.276437,0.313877,1.137022,1.018923,0.550491,-0.688145,-0.252558,1,0
4434,0.396273,-0.785868,-0.569269,-0.423817,-0.508488,-0.613845,-0.079744,-0.365393,-2.217771,-0.667321,...,-0.292935,-0.207771,-0.234416,0.528227,0.415692,-0.255429,-0.846441,-0.492986,1,0
4435,-0.330789,0.368761,-0.736246,-0.846380,-0.788138,-0.815297,0.044992,-0.930165,-2.116195,-0.356458,...,-0.011440,-0.207771,0.000567,0.699825,0.697200,0.057984,-0.846441,-0.252558,1,0
4436,0.508128,-0.995800,-0.174596,-0.697710,-0.508488,-0.362030,0.232097,-0.294796,0.495774,-1.377989,...,-0.995644,-0.524567,-0.730490,-0.160605,0.013539,-0.524068,-0.635380,-0.492986,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5534,1.291117,-0.733384,-0.220136,-1.059059,-1.021180,-0.714571,0.481570,-0.153603,0.147511,0.118386,...,0.668667,1.340150,0.601078,0.442356,0.254831,0.326624,0.050571,1.190012,1,0
5535,-0.778211,0.106346,0.189717,-0.723432,-0.694921,-0.110214,0.793411,0.340572,0.568328,0.356171,...,0.839207,1.355604,0.692460,0.755697,0.496123,0.416171,-0.160491,1.490548,1,0
5536,-0.051150,-0.103587,0.007560,-1.401907,-1.440656,-0.714571,1.042884,0.128783,0.611861,0.062331,...,1.147413,1.098046,0.836061,0.814980,0.536338,0.505717,-0.107725,1.730976,1,0
5537,0.172561,0.946076,0.265616,1.140850,0.936373,0.343054,-0.828163,0.340572,-0.505481,-0.375262,...,2.211750,2.689753,2.885632,1.502511,1.179784,2.251876,1.105880,3.233653,0,1


# Let's do one with no features removed

In [7]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(C=10**12) # have 1/C be near 0 so regularization be practically none
lr.fit(X_train, y_train)
lr.score(X_test, y_test)

0.5636856368563685

Get baseline

In [8]:
(y_test == y_train.mode()[0]).sum()/len(y_test)

np.float64(0.5582655826558266)

It does almost 1% better than the baseline.

Examine the weights

In [9]:
weights = pd.DataFrame()
weights["Feature"] = X_train.columns
weights["Weight"] = lr.coef_[0]
weights = weights.sort_values(by="Weight", ascending = False)
weights.head(10)

,Feature,Weight
32,previous_10_games_team1_average_kddiff,0.383458
39,previous_10_games_team2_std_kills,0.335058
67,previous_10_games_team2_median_kast,0.289371
70,previous_10_games_team2_range_kddiff,0.267506
31,previous_10_games_team1_median_kast,0.265528
46,previous_10_games_team2_range_deaths,0.256985
66,previous_10_games_team2_min_kast,0.190849
30,previous_10_games_team1_min_kast,0.172841
65,previous_10_games_team2_max_kast,0.166528
71,previous_10_games_team2_max_kddiff,0.159451


In [10]:
weights.tail(10)

,Feature,Weight
36,previous_10_games_team1_min_kddiff,-0.192840
40,previous_10_games_team2_range_kills,-0.205338
37,previous_10_games_team1_median_kddiff,-0.228415
35,previous_10_games_team1_max_kddiff,-0.263610
75,bestOf_5.0,-0.276337
8,previous_10_games_team1_average_deaths,-0.293996
45,previous_10_games_team2_std_deaths,-0.299128
26,previous_10_games_team1_average_kast,-0.422854
69,previous_10_games_team2_std_kddiff,-0.529913
62,previous_10_games_team2_average_kast,-0.632260


This model thinks team2 average kast, team2 std kddiff, team1 average kast, and team1 average kddiff is most predictive of the team 1 win, but I do not understand why it thinks team1 average kast being high makes them have less of a chance to win. I think this could be a collinearity issue.

# Let's retry with collinear features removed

I had ChatGPT generate me code to remove highly correlated features and only keep one of the pairs.

In [11]:
import numpy as np

X_train_base = pd.concat([train_numerical, train_cat], axis=1)
X_test_base = pd.concat([test_numerical, test_cat], axis=1)

corr = X_train_base.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

correlated_pairs = []
for row in upper.index:
    for column in upper.columns:
        value = upper.loc[row, column]
        if pd.notna(value) and value > 0.8:
            correlated_pairs.append((row, column, value))

print("Highly correlated feature pairs (abs corr > 0.8):")
for left_feature, right_feature, value in correlated_pairs:
    print(f"{left_feature} <-> {right_feature}: {value:.3f}")

features_to_keep = []
features_to_drop = []
for feature in X_train_base.columns:
    if any(corr.loc[feature, kept_feature] > 0.8 for kept_feature in features_to_keep):
        features_to_drop.append(feature)
    else:
        features_to_keep.append(feature)

print("\nFeatures elected to drop:")
print(features_to_drop)

X_train_reduced = X_train_base[features_to_keep].copy()
X_test_reduced = X_test_base[features_to_keep].copy()

features_to_keep

Highly correlated feature pairs (abs corr > 0.8):
previous_10_games_team1_average_kills <-> previous_10_games_team1_median_kills: 0.900
previous_10_games_team1_std_kills <-> previous_10_games_team1_range_kills: 0.985
previous_10_games_team1_std_kills <-> previous_10_games_team1_std_kddiff: 0.859
previous_10_games_team1_std_kills <-> previous_10_games_team1_range_kddiff: 0.841
previous_10_games_team1_range_kills <-> previous_10_games_team1_std_kddiff: 0.847
previous_10_games_team1_range_kills <-> previous_10_games_team1_range_kddiff: 0.845
previous_10_games_team1_average_deaths <-> previous_10_games_team1_max_deaths: 0.926
previous_10_games_team1_average_deaths <-> previous_10_games_team1_min_deaths: 0.907
previous_10_games_team1_average_deaths <-> previous_10_games_team1_median_deaths: 0.972
previous_10_games_team1_std_deaths <-> previous_10_games_team1_range_deaths: 0.982
previous_10_games_team1_max_deaths <-> previous_10_games_team1_median_deaths: 0.873
previous_10_games_team1_min_de

['team1_previous_10_average_map_score',
 'team2_previous_10_average_map_score',
 'previous_10_games_team1_average_kills',
 'previous_10_games_team1_std_kills',
 'previous_10_games_team1_max_kills',
 'previous_10_games_team1_min_kills',
 'previous_10_games_team1_average_deaths',
 'previous_10_games_team1_std_deaths',
 'previous_10_games_team1_average_assists',
 'previous_10_games_team1_std_assists',
 'previous_10_games_team1_max_assists',
 'previous_10_games_team1_min_assists',
 'previous_10_games_team1_average_adr',
 'previous_10_games_team1_std_adr',
 'previous_10_games_team1_max_adr',
 'previous_10_games_team1_min_adr',
 'previous_10_games_team1_median_adr',
 'previous_10_games_team1_std_kast',
 'previous_10_games_team1_max_kast',
 'previous_10_games_team1_min_kast',
 'previous_10_games_team1_median_kast',
 'previous_10_games_team1_max_kddiff',
 'previous_10_games_team1_min_kddiff',
 'previous_10_games_team1_median_kddiff',
 'previous_10_games_team2_average_kills',
 'previous_10_game

In [12]:
lr = LogisticRegression(C=10**12)
lr.fit(X_train_reduced, y_train)
lr.score(X_test_reduced, y_test)

0.5745257452574526

It performs about 2% better than baseline with collinear features removed.

In [13]:
weights = pd.DataFrame()
weights["Feature"] = X_train_reduced.columns
weights["Weight"] = lr.coef_[0]
weights = weights.sort_values(by="Weight", ascending = False)
display(weights.head(10))
display(weights.tail(10))

,Feature,Weight
2,previous_10_games_team1_average_kills,0.360707
28,previous_10_games_team2_average_deaths,0.211916
5,previous_10_games_team1_min_kills,0.157074
30,previous_10_games_team2_average_assists,0.144623
9,previous_10_games_team1_std_assists,0.134205
3,previous_10_games_team1_std_kills,0.128479
4,previous_10_games_team1_max_kills,0.125196
19,previous_10_games_team1_min_kast,0.107542
43,previous_10_games_team2_min_kddiff,0.091869
20,previous_10_games_team1_median_kast,0.083269


,Feature,Weight
1,team2_previous_10_average_map_score,-0.119814
13,previous_10_games_team1_std_adr,-0.138842
26,previous_10_games_team2_max_kills,-0.170640
44,bestOf_3.0,-0.175020
24,previous_10_games_team2_average_kills,-0.196139
22,previous_10_games_team1_min_kddiff,-0.217619
21,previous_10_games_team1_max_kddiff,-0.228962
45,bestOf_5.0,-0.229101
23,previous_10_games_team1_median_kddiff,-0.232002
6,previous_10_games_team1_average_deaths,-0.592458


This model seems to heavily value team1 average deaths, then team1 average kills, then team1 median kddiff. I do not fully understand why team2 assists is weighted positively (maybe if team 2 is getting a lot of assists and depending on each other rather than getting the kills themselves, this indicates poor individual performance). The model does seem to agree that if team 1 has an imbalance in damage output then they are less likely to win (negative weight on team1 std adr) which means a carry's presence seems to lower the chances of winning in terms of adr.

## Finally, let's do one with forward feature selection

In [19]:
from sklearn.model_selection import cross_validate
from sklearn.model_selection import TimeSeriesSplit
def SelectFeature(model, candidates, X, y):
    best_R2 = None
    best_R2_feature = None
    for candidate in candidates:
        #Get the slice of df consisting the new model with a feature added
        new_model = model.copy() # so it doesn't mutate the input array
        #Asked ChatGPT how to get a slice of columns given col labels array
        new_model.append(candidate)
        X_temp = X[new_model]
        #Get the CV R2
        lr = LogisticRegression(fit_intercept=False, C=10**10) # bias is one of our features explicitly
        cv = cross_validate(lr, X_temp, y, n_jobs=-1, cv=TimeSeriesSplit())
        test_r2 = cv["test_score"].mean()
        if(best_R2 is None):
            best_R2 = test_r2
            best_R2_feature = candidate
            continue
        if(best_R2 < test_r2):
            best_R2 = test_r2
            best_R2_feature = candidate
    return (best_R2_feature, best_R2)

In [20]:
import time
models = []
accs = []
model = ["bias"] #initialize with bias
#Score the bias only model
X_train["bias"] = 1
X_test["bias"] = 1
models.append(model.copy())
lr = LogisticRegression(fit_intercept=False, C=10**10)
result = cross_validate(lr, X_train[model], y_train, n_jobs=-1, cv=TimeSeriesSplit())
accs.append(result["test_score"].mean())
#Ask ChatGPT how to make .columns a mutable list
candidates = list(X_train.drop("bias",axis=1).columns)
i = 0
print(f"Searching over {len(candidates)} candidates.")
while len(candidates) > 0:
    i += 1
    start = time.time()
    best_feature, best_acc = SelectFeature(model,candidates,X_train,y_train)
    print(f"({i}) Candidate {best_feature} selected with total accuracy {best_acc} [{round(time.time() - start, 2)}s]")
    model.append(best_feature)
    accs.append(best_acc)
    models.append(model.copy())
    candidates.remove(best_feature)

Searching over 76 candidates.
(1) Candidate previous_10_games_team1_average_adr selected with total accuracy 0.559620596205962 [0.99s]
(2) Candidate previous_10_games_team2_average_kddiff selected with total accuracy 0.5696476964769647 [0.95s]
(3) Candidate team2_previous_10_average_map_score selected with total accuracy 0.5747967479674796 [0.94s]
(4) Candidate previous_10_games_team2_median_assists selected with total accuracy 0.5791327913279133 [0.92s]
(5) Candidate previous_10_games_team2_max_assists selected with total accuracy 0.5818428184281842 [0.92s]
(6) Candidate previous_10_games_team1_average_assists selected with total accuracy 0.5848238482384824 [0.9s]
(7) Candidate previous_10_games_team1_max_assists selected with total accuracy 0.5842818428184282 [0.9s]
(8) Candidate previous_10_games_team1_std_adr selected with total accuracy 0.5845528455284553 [0.89s]
(9) Candidate previous_10_games_team2_max_adr selected with total accuracy 0.5845528455284553 [0.88s]
(10) Candidate pr

Now let's see which model did best

In [23]:
model_df = pd.DataFrame()
model_df["Model"] = models
model_df["Validation Accuracy"] = accs
model_df.head(20)

,Model,Validation Accuracy
0,[bias],0.543631
1,"[bias, previous_10_games_team1_average_adr]",0.559621
2,"[bias, previous_10_games_team1_average_adr, pr...",0.569648
3,"[bias, previous_10_games_team1_average_adr, pr...",0.574797
4,"[bias, previous_10_games_team1_average_adr, pr...",0.579133
5,"[bias, previous_10_games_team1_average_adr, pr...",0.581843
6,"[bias, previous_10_games_team1_average_adr, pr...",0.584824
7,"[bias, previous_10_games_team1_average_adr, pr...",0.584282
8,"[bias, previous_10_games_team1_average_adr, pr...",0.584553
9,"[bias, previous_10_games_team1_average_adr, pr...",0.584553


In [25]:
best_entry = model_df.iloc[model_df["Validation Accuracy"].idxmax()]
best_entry

Model                  [bias, previous_10_games_team1_average_adr, pr...
Validation Accuracy                                             0.584824
Name: 6, dtype: object

In [26]:
best_model = best_entry["Model"]

Looks like model 6 with 6 features excluding the bias did the best, which are:

In [27]:
best_model

['bias',
 'previous_10_games_team1_average_adr',
 'previous_10_games_team2_average_kddiff',
 'team2_previous_10_average_map_score',
 'previous_10_games_team2_median_assists',
 'previous_10_games_team2_max_assists',
 'previous_10_games_team1_average_assists']

Let's see how the logistic regressor distributes the weights

In [28]:
best_estimator = LogisticRegression(C=10**10)
best_estimator.fit(X_train[best_model], y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",10000000000
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :t

In [30]:
weights = pd.DataFrame()
weights["Feature"] = X_train[best_model].columns
weights["Weight"] = best_estimator.coef_[0]
weights = weights.sort_values(by="Weight", ascending = False)
weights

,Feature,Weight
1,previous_10_games_team1_average_adr,0.232286
0,bias,0.095631
4,previous_10_games_team2_median_assists,0.033247
5,previous_10_games_team2_max_assists,0.011357
6,previous_10_games_team1_average_assists,-0.003853
3,team2_previous_10_average_map_score,-0.121073
2,previous_10_games_team2_average_kddiff,-0.194957


Looks like it most values team1 average adr, followed by team2 kddiff and team2 previous map score with the other 3 contributing slightly but not much.

In [31]:
best_estimator.score(X_test[best_model], y_test)

0.5627822944896116

This model performed worse on the test set than if we just kept all the features. I suspect this is because as the meta changes in later matches held by the test set, certain other stats become more important than the ones we determined are important at train time.

As a final note, I will not try feature engineering a higher degree polynomial since this would yield ~3000 features, which would take too long for forward feature selection to practically select out of in a reasonable amount of time on my laptop.